In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import pickle

import sys
sys.path.insert(0, 'src')
from bankit import create_data_map

years = np.arange(2008, 2016)
names = ['Attività e Passività', 'Costi e Ricavi', 'Entrate', 'Spese']


# Load Data

In [ ]:
columns = ['Denominazione', 'Dizione_Provincia', 'Dizione_Regione', 'Dizione_zona']

final = pd.read_csv('data//Anagrafe_comuni.csv', sep = ';', encoding = 'latin_1', low_memory = False).set_index('Id_Ente')

final['Data_Istituzione'] = pd.to_datetime(final['Data_Istituzione'], format='%Y-%m-%d')
final = final[final['Data_Istituzione'].dt.year < 2008]

final = final[final['Data_Cessazione'].isna()]

final = final[final['Codice_Tipologia_DLGS_118_2011'] == 'ELCOMU'][columns]

final = final.rename(columns={'Denominazione': 'Comune', 'Dizione_Provincia': 'Provincia', 'Dizione_Regione': 'Regione', 'Dizione_zona': 'Zona'})
final.index.names = ['BDAP']

final

In [ ]:
dataframes = {}

for name in names:
	
    for year in years:
		
        for category in ["CAT I", "CAT II"]:
			
            file_path = f'data//{name} {year} {category}.pkl'
            df = pd.read_pickle(file_path)  
            dataframes[f'{name}_{year}_{category}'] = df

In [ ]:
comuni = pd.read_pickle('data//comuni.pkl')
zone = pd.read_pickle('data//zona.pkl')
pop = pd.read_pickle('data//pop.pkl')
critici = pd.read_csv('data//CriticitàComuni.csv', sep = ';', low_memory = False)
for year in np.arange(2008, 2020):
	comuni[f'Target {year}'] = 0
for year in np.arange(2008, 2020):
	for idx, row in critici.iterrows():
		if int(row['Anno']) == year:
			comune = row['Comune']
			comune_2 = comune.replace('-', ' ')
			comuni.loc[(comuni['Comune'] == f'COMUNE DI {comune}') | (comuni['Comune'] == f'COMUNE DI {comune_2}'), f'Target {year}'] = 1

# Map

In [ ]:
shapefile_path = "shapes//Com01012018_WGS84.shp" 
gdf = gpd.read_file(shapefile_path, encoding="utf-8")

In [ ]:
anagrafe = pd.read_csv('data//Anagrafe_comuni.csv', sep = ';', encoding = 'latin_1', low_memory = False).set_index('Id_Ente')
anagrafe = anagrafe.loc[anagrafe['Codice_Tipologia_DLGS_118_2011'] == 'ELCOMU']

In [ ]:
file1 = 'models_CAT I_final.pickle'
models = {}

with open(file1, 'rb') as f1:
    models['CAT I'] = pickle.load(f1)
model = models['CAT I'][3][8]['CNN']

category = 'CAT I'
features = names[2:4]
mode = 'pop'
add = ['pop', 'zone']

X, y = create_data_map(dataframes, comuni, pop, category, features, 8, 3, mode, add, zone=zone)
proba = model.predict(X)

In [ ]:
t = np.log(proba + 1e-6)
x = (t - np.min(t)) / (np.max(t) - np.min(t))

In [ ]:
geo = pd.DataFrame(index = final.index)
geo["Codice_Istat"] = anagrafe.loc[geo.index, "Codice_ISTAT_Comune"]
geo['Probs'] = x
geo['Truth'] = y

In [ ]:
gdf = gdf[["geometry", "PRO_COM"]].rename(columns={"PRO_COM": "Codice_Istat"})

gdf = gdf.merge(geo[["Codice_Istat", "Probs", "Truth"]], on="Codice_Istat", how="left")

gdf["Probs"] = gdf.apply(lambda row: -1 if row["Truth"] == 1 else row["Probs"], axis=1)

gdf = gdf.drop(columns=["Truth"])

In [ ]:
save = 0

na_color = "gray"
negative_one_color = "black" 

cmap = plt.cm.coolwarm 
norm = mcolors.Normalize(vmin=0, vmax=1)

gdf["color"] = gdf["Probs"].apply(lambda x: na_color if np.isnan(x) else (negative_one_color if x == -1 else cmap(norm(x))))

fig, ax = plt.subplots(1, 1, figsize=(10, 9))

gdf.plot(ax=ax, color=gdf["color"], edgecolor=None)

legend_elements = [
    Patch(facecolor=na_color, edgecolor='none', label='Missing Municipalities'),
    Patch(facecolor=negative_one_color, edgecolor='none', label='Municipalities in Financial Distress'),
    Patch(facecolor=cmap(0.0), edgecolor='none', label='Low Probability'),
    Patch(facecolor=cmap(1.0), edgecolor='none', label='High Probability'),
]
ax.legend(handles=legend_elements, title="Legend", fontsize=10, loc='center left', bbox_to_anchor=(0.8, 0.8))

plt.title("Financial Distress Probability ", fontsize=16)

ax.axis("off")

plt.tight_layout()

if save:
	plt.savefig('map.png')
plt.show()